In [3]:
print(1/2)

0.5


In [5]:
# pip install httpx
import httpx, json
from datetime import date

URL = "https://highways-rows-colleagues-balance.trycloudflare.com"
TOKEN = "IvrmsjuKeCyYwJheMAVPXP8ZF_zCIrmq"

# Schema ép đầu ra: Ollama dùng schema này để ràng buộc JSON
SCHEMA = {
    "type": "object",
    "properties": {
        "keywords":     {"type": "string"},
        "category":     {"type": ["string", "null"]},
        "tags":         {"type": "array", "items": {"type": "string"}},
        "difficulty":   {"type": ["string", "null"], "enum": ["easy", "medium", "hard", None]},
        "search_in":    {"type": "array", "items": {"type": "string", "enum": ["question", "options", "all"]}},
        "created_from": {"type": ["string", "null"]},
        "created_to":   {"type": ["string", "null"]},
    },
    "required": ["keywords", "category", "tags", "difficulty",
                 "search_in", "created_from", "created_to"],
}

SYSTEM = f"""Bạn chuyển câu tìm kiếm của Admin thành bộ lọc JSON cho ngân hàng câu hỏi trắc nghiệm.
Hôm nay là {date.today().isoformat()}.
Quy tắc:
- category: tên bộ đề/môn nếu có (ví dụ "TOEIC"), không có thì null.
- tags: các nhãn phụ như "Part 5", "Part 6"; không có thì [].
- difficulty: chỉ easy | medium | hard (dễ=easy, trung bình=medium, khó=hard), không nói thì null.
- search_in: "question" nếu nhắc dòng câu hỏi/nội dung câu hỏi; "options" nếu nhắc đáp án; mặc định ["all"].
- keywords: chỉ phần từ khóa còn lại, KHÔNG lặp lại thông tin đã đưa vào các trường khác. Không còn thì "".
- Ngày tương đối đổi sang yyyy-mm-dd; không nhắc thì null.
- KHÔNG bịa giá trị không có trong câu. Chỉ trả JSON.

Ví dụ:
Câu: "câu dễ ngữ pháp tạo tuần trước ở đáp án"
JSON: {{"keywords":"ngữ pháp","category":null,"tags":[],"difficulty":"easy","search_in":["options"],"created_from":null,"created_to":null}}
"""

def search_to_filter(text: str) -> dict:
    payload = {
        "model": "qwen2.5:14b-instruct-q4_K_M",
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
        "format": SCHEMA,          # nếu Ollama báo lỗi, đổi thành "json"
        "stream": False,
        "options": {"num_ctx": 4096, "temperature": 0},
    }
    r = httpx.post(f"{URL}/api/chat",
                   headers={"Authorization": f"Bearer {TOKEN}"},
                   json=payload, timeout=180)
    r.raise_for_status()
    d = r.json()
    print(f"Thời gian: {d['total_duration']/1e9:.1f}s | "
          f"{d['eval_count']/(d['eval_duration']/1e9):.1f} token/s")
    return json.loads(d["message"]["content"])

if __name__ == "__main__":
    q = "Bộ đề kiểm tra TOEIC cấp độ khó ở dòng câu hỏi part 5."
    print(json.dumps(search_to_filter(q), ensure_ascii=False, indent=2))

Thời gian: 5.4s | 11.8 token/s
{
  "keywords": "",
  "category": "TOEIC",
  "tags": [
    "Part 5"
  ],
  "difficulty": "hard",
  "search_in": [
    "question"
  ],
  "created_from": null,
  "created_to": null
}


In [4]:
# pip install httpx
import httpx

URL = "https://highways-rows-colleagues-balance.trycloudflare.com"
TOKEN = "IvrmsjuKeCyYwJheMAVPXP8ZF_zCIrmq"

payload = {
    "model": "qwen2.5:14b-instruct-q4_K_M",
    "messages": [
        {"role": "system", "content":
         "Bạn là bộ kiểm tra chính tả tiếng Việt và tiếng Anh. Chỉ trả về JSON: "
         '{"has_error": bool, "errors": [{"original": str, "suggestion": str, "reason": str}]}'},
        {"role": "user", "content": "Hãy sửa chính tả: He wnt homework on next year."},
    ],
    "format": "json",
    "stream": False,
    "options": {"num_ctx": 4096, "temperature": 0},
}

r = httpx.post(f"{URL}/api/chat",
               headers={"Authorization": f"Bearer {TOKEN}"},
               json=payload, timeout=180)
r.raise_for_status()
data = r.json()

print(data["message"]["content"])
print(f"Tốc độ: {data['eval_count'] / (data['eval_duration'] / 1e9):.1f} token/s")
print(f"Tổng thời gian: {data['total_duration'] / 1e9:.1f} s")

{"has_error": true, "errors": [{"original": "wnt", "suggestion": "wants", "reason": "Missing letters"}]}
Tốc độ: 13.8 token/s
Tổng thời gian: 2.5 s
